# ============================================
# Cell 1: Import libraries and define global variables
# ============================================

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap
from matplotlib.pyplot import cm
import seaborn as sns

# Assume the following modules provide required functions:
from ozone_extremes_leap import *   # e.g. calc_parc_o3, find_ozone_extremes_baseonpressure
from Weiji_module import *          # e.g. analyse_ozone_extremes
# Global file paths and parameters
BASE_DIR = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart'
PSL_FILE = os.path.join(BASE_DIR, 'BWCN.e122.f19_g16.PSL.merged.nc')
O3_FILE  = os.path.join(BASE_DIR, 'BWCN.e122.f19_g16.merged.nc')

# ============================================
# Cell 2: Read data and preprocess for ozone analysis
# ============================================

In [ ]:
# Read surface pressure data and compute daily climatology
ds_surf = xr.open_dataset(PSL_FILE)
psl = ds_surf['PSL']
psl_clim = psl.groupby('time.dayofyear').mean()

# Read ozone data
ds_ozone = xr.open_dataset(O3_FILE)
ozone = ds_ozone['O3']

# Compute partial ozone columns using the imported function calc_parc_o3
o3_30_70 = calc_parc_o3(ozone, 30, 70)
o3_30_100 = calc_parc_o3(ozone, 30, 100)

# ============================================
# Cell 3: Define helper function for spatial average
# ============================================

In [ ]:
def spatial_average(data, lat_start=60, lat_end=90):
    """
    Compute the weighted spatial average (using cosine of latitude).
    """
    zonal_mean = data.mean(dim='lon')
    subset = zonal_mean.sel(lat=slice(lat_start, lat_end))
    weights = np.cos(np.deg2rad(subset.lat))
    return subset.weighted(weights).mean(dim='lat')

# Compute spatial averages for tropical (0–30°) and polar (60–90°) regions
o3_30_70_tropical = spatial_average(o3_30_70, lat_start=0, lat_end=30)
o3_30_70_polar    = spatial_average(o3_30_70, lat_start=60, lat_end=90)
o3_30_100_tropical = spatial_average(o3_30_100, lat_start=0, lat_end=30)
o3_30_100_polar    = spatial_average(o3_30_100, lat_start=60, lat_end=90)

# ============================================
# Cell 4: Define plotting functions for ozone evolution and surface response
# ============================================

In [ ]:
def plot_o3_case(data, case_range='30-70', region='polar', extreme_years=10, years=39, model='WACCM', level='plev'):
    """
    Plot ozone evolution with extreme-year analysis.
    Parameters:
      data         : xarray DataArray (e.g. spatially averaged ozone)
      case_range   : Pressure range string (e.g. '30-70')
      region       : 'tropical' or 'polar'
      extreme_years: Number of extreme years to use
      years        : Total number of years
      model        : Model name
      level        : Pressure level variable name
    Returns:
      (fig, ax): The matplotlib figure and axis.
    """
    fig, ax = plt.subplots(figsize=(13,10))
    # Extract top and bottom pressure from case_range
    p_top, p_bottom = map(int, case_range.split('-'))
    # Get ozone extremes (function provided by imported module)
    (o3_highest, o3_lowest, o3_lowest_date, o3_highest_date,
     o3_lowest_index, o3_highest_index, o3_intersect, o3_years) = \
         find_ozone_extremes_baseonpressure(ds_ozone, 'O3', level, years, extreme_years, model, p_top=p_top, p_bottom=p_bottom)
    
    # Perform surface pressure anomaly analysis (using imported function)
    (min_slp, xpt, ypt, anomalies_slp, significance_slp) = \
         analyse_ozone_extremes(ds_surf, 'PSL', extreme_years, True, o3_highest_date, o3_lowest_date, o3_lowest_index, o3_years, model)
    
    # For simplicity, here we plot the climatology of the input data
    clim = data.groupby("time.dayofyear").mean("time")
    ax.plot(np.arange(len(clim)), clim, color='grey', label='Climatology')
    
    ax.set_title(f'Ozone Evolution ({region.capitalize()}, {case_range} hPa)')
    ax.set_xlabel('Day of Year')
    ax.set_ylabel('Partial Ozone Column (DU)')
    ax.legend()
    
    save_base = f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_10extr_{case_range.replace("-","to")}_{region}'
    plt.savefig(save_base + '.pdf')
    plt.savefig(save_base + '.png')
    return fig, ax

def plot_surface_response(ds_surf, ds_ozone, case_range='30-70', extreme_years=10, years=39, model='WACCM', level='plev'):
    """
    Plot the mean surface pressure response associated with ozone extremes.
    """
    p_top, p_bottom = map(int, case_range.split('-'))
    (o3_highest, o3_lowest, o3_lowest_date, o3_highest_date,
     o3_lowest_index, o3_highest_index, o3_intersect, o3_years) = \
         find_ozone_extremes_baseonpressure(ds_ozone, 'O3', level, years, extreme_years, model, p_top=p_top, p_bottom=p_bottom)
    
    (min_slp, xpt, ypt, anomalies_slp, significance_slp) = \
         analyse_ozone_extremes(ds_surf, 'PSL', extreme_years, True, o3_highest_date, o3_lowest_date, o3_lowest_index, o3_years, model)
    
    fig = plt.figure(figsize=(10,8))
    ax = plt.gca()
    m = Basemap(projection='ortho', lat_0=90, lon_0=0, resolution='l', ax=ax)
    m.drawcoastlines(linewidth=0.3)
    m.drawmeridians(np.arange(0,360,60), linewidth=0.3)
    m.drawparallels(np.arange(-90,90,30), linewidth=0.3)
    
    levels = np.linspace(-4,4,16)
    cs = m.contourf(xpt, ypt, min_slp/100, cmap='RdBu_r', levels=levels, extend='both')
    plt.title(f'Mean Surface Response ({case_range} hPa, n={extreme_years})')
    plt.colorbar(cs)
    
    save_base = f'/home/weiji/restart_exam/plots/surface_response_{case_range.replace("-","to")}_n{extreme_years}'
    plt.savefig(save_base + '.pdf')
    plt.savefig(save_base + '.png')
    return fig

def plot_o3_evolution(data_base, clim_data, jan_data, feb_data, mar_data, pressure_range='30-70'):
    """
    Plot ozone evolution for base, climatology, and perturbation (Jan, Feb, Mar) data.
    """
    fig, ax = plt.subplots(figsize=(12,8))
    ax.plot(np.arange(len(data_base)), data_base, color='black', linewidth=3, label='Base')
    ax.plot(np.arange(len(clim_data)), clim_data, color='black', linestyle=':', linewidth=3, label='Climatology')
    
    # Plot Jan, Feb and Mar perturbation data (using offsets in x-axis)
    ax.plot(np.arange(len(jan_data.time)), jan_data.mean(dim='member'), color='forestgreen', linewidth=3, label='Jan')
    ax.plot(np.arange(31, 31+len(feb_data.time)), feb_data.mean(dim='member'), color='royalblue', linewidth=3, label='Feb')
    ax.plot(np.arange(59, 59+len(mar_data.time)), mar_data.mean(dim='member'), color='hotpink', linewidth=3, label='Mar')
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'])
    ax.set_ylabel(f'Partial Ozone Column, {pressure_range} hPa (DU)')
    ax.legend()
    
    save_base = f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_0008_{pressure_range.replace("-","to")}'
    plt.savefig(save_base + '.pdf')
    plt.savefig(save_base + '.png')
    return fig, ax

def plot_o3_evolution_mar_types(base_data, mar_small, mar_large, mar_heating, pressure_range='30-70'):
    """
    Plot March ozone evolution for three perturbation types.
    """
    fig, ax = plt.subplots(figsize=(12,8))
    ax.plot(np.arange(len(base_data)), base_data, color='black', linewidth=3, label='Base')
    ax.plot(np.arange(len(base_data)), base_data, color='black', linestyle=':', linewidth=3, label='Climatology')
    
    x_mar = np.arange(59, 59+len(mar_small.time))
    ax.plot(x_mar, mar_small.mean(dim='member'), color='crimson', linewidth=3, label='Mar, small pert')
    ax.fill_between(x_mar, mar_small.min(dim='member'), mar_small.max(dim='member'), color='crimson', alpha=0.3)
    
    x_mar2 = np.arange(59, 59+len(mar_large.time))
    ax.plot(x_mar2, mar_large.mean(dim='member'), color='indigo', linewidth=3, label='Mar, large pert')
    ax.fill_between(x_mar2, mar_large.min(dim='member'), mar_large.max(dim='member'), color='indigo', alpha=0.3)
    
    x_mar3 = np.arange(59, 59+len(mar_heating.time))
    ax.plot(x_mar3, mar_heating.mean(dim='member'), color='teal', linewidth=3, label='Mar, heating pert')
    ax.fill_between(x_mar3, mar_heating.min(dim='member'), mar_heating.max(dim='member'), color='teal', alpha=0.3)
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'])
    ax.set_ylabel(f'Partial Ozone Column, {pressure_range} hPa (DU)')
    ax.legend()
    
    save_base = f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_0008_Mar_{pressure_range.replace("-","to")}'
    plt.savefig(save_base + '.pdf', bbox_inches="tight")
    plt.savefig(save_base + '.png')
    return fig, ax

# ============================================
# Cell 5: Preprocess 008 ozone data for perturbation analysis
# ============================================

In [ ]:
# For demonstration, we filter the 30-70 polar data for months January–June and select year 8.
o3_30_70_polar_Jun = o3_30_70_polar.sel(time=o3_30_70_polar.time.dt.month.isin([1,2,3,4,5,6]))
o3_30_70_008_polar = o3_30_70_polar_Jun.sel(time=o3_30_70_polar_Jun.time.dt.year==8)
clim_30_70_polar = o3_30_70_polar_Jun.groupby('time.dayofyear').mean()

# For perturbation experiments, assume the following monthly subsets (dummy selections)
var_J_30_70 = o3_30_70_008_polar.sel(time=o3_30_70_008_polar.time.dt.month==1)
var_F_30_70 = o3_30_70_008_polar.sel(time=o3_30_70_008_polar.time.dt.month==2)
var_M_30_70 = o3_30_70_008_polar.sel(time=o3_30_70_008_polar.time.dt.month==3)

# ============================================
# Cell 6: Plot 008 ozone evolution and March perturbation comparison
# ============================================

In [ ]:
# Plot overall ozone evolution
fig1, ax1 = plot_o3_evolution(o3_30_70_008_polar, clim_30_70_polar, var_J_30_70, var_F_30_70, var_M_30_70, pressure_range='30-70')

# For March perturbations, assume that the three different perturbation datasets are available.
# Here, for demonstration, we use the same dataset as dummy data.
fig2, ax2 = plot_o3_evolution_mar_types(o3_30_70_008_polar, var_M_30_70, var_M_30_70, var_M_30_70, pressure_range='30-70')

# (Additional surface response plots could be generated by calling plot_surface_response() as needed.)

plt.close('all')